In [1]:
import pandas as pd
import numpy as np

# 1. Dataset fayllarini yuklaymiz (u.data va u.item)
data_url = 'https://files.grouplens.org/datasets/movielens/ml-100k/u.data'
items_url = 'https://files.grouplens.org/datasets/movielens/ml-100k/u.item'

# u.data: user_id | item_id | rating | timestamp
ratings_cols = ['user_id', 'movie_id', 'rating', 'timestamp']
ratings = pd.read_csv(data_url, sep='\t', names=ratings_cols, encoding='latin-1')

# u.item: movie_id | movie_title | ...
items_cols = ['movie_id', 'movie_title'] + [str(i) for i in range(22)] # Qolgan 22 ta ustun bizga shart emas
movies = pd.read_csv(items_url, sep='|', names=items_cols, usecols=['movie_id', 'movie_title'], encoding='latin-1')

# 2. Reytinglar va Kino nomlarini birlashtiramiz
df = pd.merge(ratings, movies, on='movie_id')
display(df.head())

,user_id,movie_id,rating,timestamp,movie_title
0,196,242,3,881250949,Kolya (1996)
1,186,302,3,891717742,L.A. Confidential (1997)
2,22,377,1,878887116,Heavyweights (1994)
3,244,51,2,880606923,Legends of the Fall (1994)
4,166,346,1,886397596,Jackie Brown (1997)


SVD modelini yaratish va o'qitish

In [2]:
!pip install "numpy<2.0"
!pip install scikit-surprise

In [4]:
!pip install "numpy<2.0" -q
!pip install scikit-surprise -q

from surprise import SVD, Dataset, Reader
from surprise.model_selection import cross_validate

reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['user_id', 'movie_id', 'rating']], reader)

svd = SVD()

# 4. Modelni tekshirib ko'ramiz (Cross-validation orqali)
# Bu model qanchalik aniq ishlayotganini (RMSE) ko'rsatadi
cv_results = cross_validate(svd, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

trainset = data.build_full_trainset()
svd.fit(trainset)

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.9345  0.9290  0.9329  0.9405  0.9449  0.9363  0.0057  
MAE (testset)     0.7327  0.7332  0.7382  0.7436  0.7459  0.7387  0.0053  
Fit time          1.87    1.26    1.31    1.27    1.31    1.41    0.23    
Test time         0.16    0.20    0.10    0.20    0.10    0.15    0.05    


Prediction bashorat

In [7]:
# Bitta foydalanuvchi (user_id) va bitta kino (movie_id) ni tanlaymiz
user_id = 1
movie_id = 313
prediction = svd.predict(uid=user_id, iid=movie_id)
print(f"Foydalanuvchi (ID: {user_id}) ushbu kinoga (ID: {movie_id}) taxminan {prediction.est:.2f} ball beradi!")

Foydalanuvchi (ID: 1) ushbu kinoga (ID: 313) taxminan 4.57 ball beradi!


Foydalanuvchi uchun Top-10 Tavsiyalarni ishlab chiqish

In [13]:
user_id = 1
all_movie_ids = df['movie_id'].unique()
watched_movie_ids = df[df['user_id'] == user_id]['movie_id'].unique()

unseen_movies = [mid for mid in all_movie_ids if mid not in watched_movie_ids]

predictions = [svd.predict(uid=user_id, iid=mid) for mid in unseen_movies]

predictions.sort(key=lambda x: x.est, reverse=True)

top_10 = predictions[:10]
print(f"Foydalanuvchi (ID: {user_id}) uchun Top-10 Tavsiyalar:\n" + "-"*50)

for i, pred in enumerate(top_10, 1):
    # Kino ID siga qarab uning nomini jadvaldan topib kelish
    movie_title = df[df['movie_id'] == pred.iid]['movie_title'].iloc[0]
    print(f"{i}. {movie_title} (Kutilayotgan ball: {pred.est:.2f} / 5.00)")

Foydalanuvchi (ID: 1) uchun Top-10 Tavsiyalar:
--------------------------------------------------
1. One Flew Over the Cuckoo's Nest (1975) (Kutilayotgan ball: 4.98 / 5.00)
2. Casablanca (1942) (Kutilayotgan ball: 4.96 / 5.00)
3. Secrets & Lies (1996) (Kutilayotgan ball: 4.91 / 5.00)
4. Close Shave, A (1995) (Kutilayotgan ball: 4.87 / 5.00)
5. Lawrence of Arabia (1962) (Kutilayotgan ball: 4.75 / 5.00)
6. Ulee's Gold (1997) (Kutilayotgan ball: 4.71 / 5.00)
7. Ran (1985) (Kutilayotgan ball: 4.70 / 5.00)
8. Annie Hall (1977) (Kutilayotgan ball: 4.66 / 5.00)
9. L.A. Confidential (1997) (Kutilayotgan ball: 4.65 / 5.00)
10. Schindler's List (1993) (Kutilayotgan ball: 4.63 / 5.00)


In [14]:
# Bashoratlarni endi ENG PASTIDAN yuqoriga qarab tartiblaymiz (reverse=False qildik)
predictions.sort(key=lambda x: x.est, reverse=False)
bottom_10 = predictions[:10]

print("\nFoydalanuvchi (ID: 1) ga UMUMan YOQMAYDIGAN Top-10 kino:\n" + "-"*50)
for i, pred in enumerate(bottom_10, 1):
    movie_title = df[df['movie_id'] == pred.iid]['movie_title'].iloc[0]
    print(f"{i}. {movie_title} (Kutilayotgan ball: {pred.est:.2f} / 5.00)")


Foydalanuvchi (ID: 1) ga UMUMan YOQMAYDIGAN Top-10 kino:
--------------------------------------------------
1. Rocket Man (1997) (Kutilayotgan ball: 1.26 / 5.00)
2. Mortal Kombat: Annihilation (1997) (Kutilayotgan ball: 1.48 / 5.00)
3. Leave It to Beaver (1997) (Kutilayotgan ball: 1.61 / 5.00)
4. Lawnmower Man 2: Beyond Cyberspace (1996) (Kutilayotgan ball: 1.68 / 5.00)
5. Free Willy 3: The Rescue (1997) (Kutilayotgan ball: 1.70 / 5.00)
6. Barb Wire (1996) (Kutilayotgan ball: 1.73 / 5.00)
7. Home Alone 3 (1997) (Kutilayotgan ball: 1.75 / 5.00)
8. Beautician and the Beast, The (1997) (Kutilayotgan ball: 1.78 / 5.00)
9. McHale's Navy (1997) (Kutilayotgan ball: 1.78 / 5.00)
10. Island of Dr. Moreau, The (1996) (Kutilayotgan ball: 1.81 / 5.00)
